In [3]:

import pandas as pd
import numpy as np



In [4]:
# Cargamos el dataset modelable a 10 segundos

df = pd.read_csv("data_limpio/dataset_modelable_10s.csv")

df["Time"] = pd.to_datetime(df["Time"])

df.head()

,Time,LT411,SP_LT411,LV411,DT412,INT_P101,FQC400_1,SP_VAPOR,FV400_1,PT442,SP_PT442,PV442,TT413,TT415,LT426,SP_LT426,ACCION_PCT02,FT428,PIT410,PIT414
0,2021-11-05 08:00:00,58.142360,60.0,100.0,1229.492151,58.408563,3054.785938,69.0,59.5,0.154167,0.14,0.0,80.976565,81.82292,50.439814,50.0,18.861438,6.266928,0.326085,0.579514
1,2021-11-05 08:00:10,57.962962,60.0,100.0,1229.049451,58.272569,3052.526001,69.0,59.5,0.154282,0.14,0.0,80.954863,81.81424,50.422452,50.0,19.102801,6.293403,0.326181,0.579456
2,2021-11-05 08:00:20,57.323494,60.0,100.0,1229.088501,58.234953,3038.494727,69.0,59.5,0.154225,0.14,0.0,80.946182,81.81424,50.434027,50.0,19.315793,6.325521,0.326403,0.579051
3,2021-11-05 08:00:30,57.335069,60.0,100.0,1228.762976,58.463541,3031.794189,69.0,59.5,0.154282,0.14,0.0,80.946182,81.81424,50.422452,50.0,19.470506,6.357205,0.326340,0.578588
4,2021-11-05 08:00:40,57.835647,60.0,100.0,1229.453088,58.159721,3036.406226,69.0,59.5,0.154485,0.14,0.0,80.946182,81.81424,50.428239,50.0,19.712129,6.391927,0.326403,0.578125


In [7]:
# Definimos cortes temporales para train, validación y test

inicio_train = pd.Timestamp("2021-11-05 08:00:00")
fin_train = pd.Timestamp("2021-11-06 07:59:59")

inicio_valid = pd.Timestamp("2021-11-06 08:00:00")
fin_valid = pd.Timestamp("2021-11-07 07:59:59")

inicio_test = pd.Timestamp("2021-11-07 08:00:00")
fin_test = df["Time"].max()

In [8]:
# Asignamos cada fila a train, validación o test

df["bloque"] = "sin_asignar"

df.loc[(df["Time"] >= inicio_train) & (df["Time"] <= fin_train), "bloque"] = "train"
df.loc[(df["Time"] >= inicio_valid) & (df["Time"] <= fin_valid), "bloque"] = "validacion"
df.loc[(df["Time"] >= inicio_test) & (df["Time"] <= fin_test), "bloque"] = "test"

df["bloque"].value_counts()

bloque
train         8640
validacion    8640
test          5790
Name: count, dtype: int64

In [9]:
# Revisamos inicio, fin y filas por bloque

df.groupby("bloque").agg(
    inicio=("Time", "min"),
    fin=("Time", "max"),
    filas=("Time", "count")
)

,inicio,fin,filas
bloque,,,
test,2021-11-07 08:00:00,2021-11-08 00:04:50,5790
train,2021-11-05 08:00:00,2021-11-06 07:59:50,8640
validacion,2021-11-06 08:00:00,2021-11-07 07:59:50,8640


In [10]:
# Definimos la variable objetivo

target = "LT411"

In [11]:
# Definimos variables excluidas del Modelo A por riesgo de leakage o circularidad

variables_excluidas_modelo_A = [
    "LT411",
    "LV411",
    "SP_LT411"
]

In [12]:
# Definimos variables para el Modelo A limpio

variables_modelo_A = [
    "DT412",
    "INT_P101",
    "FQC400_1",
    "SP_VAPOR",
    "FV400_1",
    "PT442",
    "SP_PT442",
    "PV442",
    "TT413",
    "TT415",
    "LT426",
    "SP_LT426",
    "ACCION_PCT02",
    "FT428",
    "PIT410",
    "PIT414"
]

In [14]:
# quitamos los set points de las variables del modelo A limpio

for columna in setpoints:
    print(columna, "valores únicos:", df[columna].nunique())

SP_VAPOR valores únicos: 1
SP_PT442 valores únicos: 1
SP_LT426 valores únicos: 1


In [15]:
# Quitamos setpoints constantes del Modelo A

variables_modelo_A = [
    "DT412",
    "INT_P101",
    "FQC400_1",
    "FV400_1",
    "PT442",
    "PV442",
    "TT413",
    "TT415",
    "LT426",
    "ACCION_PCT02",
    "FT428",
    "PIT410",
    "PIT414"
]

In [16]:
# Creamos el dataset base para feature engineering

columnas_base = ["Time", "bloque", target] + variables_modelo_A

df_base = df[columnas_base].copy()

df_base.head()

,Time,bloque,LT411,DT412,INT_P101,FQC400_1,FV400_1,PT442,PV442,TT413,TT415,LT426,ACCION_PCT02,FT428,PIT410,PIT414
0,2021-11-05 08:00:00,train,58.142360,1229.492151,58.408563,3054.785938,59.5,0.154167,0.0,80.976565,81.82292,50.439814,18.861438,6.266928,0.326085,0.579514
1,2021-11-05 08:00:10,train,57.962962,1229.049451,58.272569,3052.526001,59.5,0.154282,0.0,80.954863,81.81424,50.422452,19.102801,6.293403,0.326181,0.579456
2,2021-11-05 08:00:20,train,57.323494,1229.088501,58.234953,3038.494727,59.5,0.154225,0.0,80.946182,81.81424,50.434027,19.315793,6.325521,0.326403,0.579051
3,2021-11-05 08:00:30,train,57.335069,1228.762976,58.463541,3031.794189,59.5,0.154282,0.0,80.946182,81.81424,50.422452,19.470506,6.357205,0.326340,0.578588
4,2021-11-05 08:00:40,train,57.835647,1229.453088,58.159721,3036.406226,59.5,0.154485,0.0,80.946182,81.81424,50.428239,19.712129,6.391927,0.326403,0.578125


In [17]:
# Definimos lags y ventanas rolling para el primer Modelo A

lags = [1, 2, 3, 6]

ventanas_rolling = [3, 6]

## Decisión sobre lags y rolling

Para el primer Modelo A se usan lags cortos: 10s, 20s, 30s y 1min.

No se incluyen inicialmente lags de 5min o 15min para evitar añadir demasiada memoria temporal y reducir el riesgo de sobreajuste.

Las rolling iniciales serán de 30s y 1min, manteniendo una ventana corta coherente con la dinámica observada en EDA_10s.

Más adelante se podrá comparar contra una versión ampliada con lags de 5min y 15min si el modelo lo necesita.

In [18]:
# Definimos variables a las que aplicaremos lags

variables_lag = variables_modelo_A.copy()

variables_lag

['DT412',
 'INT_P101',
 'FQC400_1',
 'FV400_1',
 'PT442',
 'PV442',
 'TT413',
 'TT415',
 'LT426',
 'ACCION_PCT02',
 'FT428',
 'PIT410',
 'PIT414']

In [19]:
# Definimos variables principales para rolling features

variables_rolling = [
    "DT412",
    "INT_P101",
    "FQC400_1",
    "PT442",
    "TT413",
    "TT415",
    "PIT410",
    "PIT414"
]

variables_rolling

['DT412',
 'INT_P101',
 'FQC400_1',
 'PT442',
 'TT413',
 'TT415',
 'PIT410',
 'PIT414']

In [20]:
# Creamos una función simple para generar lags y rolling por bloque

def crear_features_temporales(datos):
    datos = datos.sort_values("Time").copy()
    
    for variable in variables_lag:
        for lag in lags:
            datos[variable + "_lag_" + str(lag)] = datos[variable].shift(lag)
    
    for variable in variables_rolling:
        for ventana in ventanas_rolling:
            datos[variable + "_roll_mean_" + str(ventana)] = datos[variable].rolling(window=ventana).mean()
            datos[variable + "_roll_std_" + str(ventana)] = datos[variable].rolling(window=ventana).std()
    
    return datos

In [21]:
# Creamos features temporales separando train, validación y test

df_train_feat = crear_features_temporales(df_base[df_base["bloque"] == "train"])

df_valid_feat = crear_features_temporales(df_base[df_base["bloque"] == "validacion"])

df_test_feat = crear_features_temporales(df_base[df_base["bloque"] == "test"])

In [22]:
# Unimos los bloques con features temporales

df_features = pd.concat([df_train_feat, df_valid_feat, df_test_feat], axis=0)

df_features = df_features.sort_values("Time").reset_index(drop=True)

df_features.head()

,Time,bloque,LT411,DT412,INT_P101,FQC400_1,FV400_1,PT442,PV442,TT413,TT415,LT426,ACCION_PCT02,FT428,PIT410,PIT414,DT412_lag_1,DT412_lag_2,DT412_lag_3,DT412_lag_6,INT_P101_lag_1,INT_P101_lag_2,INT_P101_lag_3,INT_P101_lag_6,FQC400_1_lag_1,FQC400_1_lag_2,FQC400_1_lag_3,FQC400_1_lag_6,FV400_1_lag_1,FV400_1_lag_2,FV400_1_lag_3,FV400_1_lag_6,PT442_lag_1,PT442_lag_2,PT442_lag_3,PT442_lag_6,PV442_lag_1,PV442_lag_2,PV442_lag_3,PV442_lag_6,TT413_lag_1,TT413_lag_2,TT413_lag_3,TT413_lag_6,TT415_lag_1,TT415_lag_2,TT415_lag_3,TT415_lag_6,LT426_lag_1,LT426_lag_2,LT426_lag_3,LT426_lag_6,ACCION_PCT02_lag_1,ACCION_PCT02_lag_2,ACCION_PCT02_lag_3,ACCION_PCT02_lag_6,FT428_lag_1,FT428_lag_2,FT428_lag_3,FT428_lag_6,PIT410_lag_1,PIT410_lag_2,PIT410_lag_3,PIT410_lag_6,PIT414_lag_1,PIT414_lag_2,PIT414_lag_3,PIT414_lag_6,DT412_roll_mean_3,DT412_roll_std_3,DT412_roll_mean_6,DT412_roll_std_6,INT_P101_roll_mean_3,INT_P101_roll_std_3,INT_P101_roll_mean_6,INT_P101_roll_std_6,FQC400_1_roll_mean_3,FQC400_1_roll_std_3,FQC400_1_roll_mean_6,FQC400_1_roll_std_6,PT442_roll_mean_3,PT442_roll_std_3,PT442_roll_mean_6,PT442_roll_std_6,TT413_roll_mean_3,TT413_roll_std_3,TT413_roll_mean_6,TT413_roll_std_6,TT415_roll_mean_3,TT415_roll_std_3,TT415_roll_mean_6,TT415_roll_std_6,PIT410_roll_mean_3,PIT410_roll_std_3,PIT410_roll_mean_6,PIT410_roll_std_6,PIT414_roll_mean_3,PIT414_roll_std_3,PIT414_roll_mean_6,PIT414_roll_std_6
0,2021-11-05 08:00:00,train,58.142360,1229.492151,58.408563,3054.785938,59.5,0.154167,0.0,80.976565,81.82292,50.439814,18.861438,6.266928,0.326085,0.579514,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-11-05 08:00:10,train,57.962962,1229.049451,58.272569,3052.526001,59.5,0.154282,0.0,80.954863,81.81424,50.422452,19.102801,6.293403,0.326181,0.579456,1229.492151,NaN,NaN,NaN,58.408563,NaN,NaN,NaN,3054.785938,NaN,NaN,NaN,59.5,NaN,NaN,NaN,0.154167,NaN,NaN,NaN,0.0,NaN,NaN,NaN,80.976565,NaN,NaN,NaN,81.82292,NaN,NaN,NaN,50.439814,NaN,NaN,NaN,18.861438,NaN,NaN,NaN,6.266928,NaN,NaN,NaN,0.326085,NaN,NaN,NaN,0.579514,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-11-05 08:00:20,train,57.323494,1229.088501,58.234953,3038.494727,59.5,0.154225,0.0,80.946182,81.81424,50.434027,19.315793,6.325521,0.326403,0.579051,1229.049451,1229.492151,NaN,NaN,58.272569,58.408563,NaN,NaN,3052.526001,3054.785938,NaN,NaN,59.5,59.5,NaN,NaN,0.154282,0.154167,NaN,NaN,0.0,0.0,NaN,NaN,80.954863,80.976565,NaN,NaN,81.81424,81.82292,NaN,NaN,50.422452,50.439814,NaN,NaN,19.102801,18.861438,NaN,NaN,6.293403,6.266928,NaN,NaN,0.326181,0.326085,NaN,NaN,0.579456,0.579514,NaN,NaN,1229.210034,0.245099,NaN,NaN,58.305362,0.091333,NaN,NaN,3048.602222,8.825980,NaN,NaN,0.154225,0.000058,NaN,NaN,80.959203,1.564940e-02,NaN,NaN,81.817133,5.011819e-03,NaN,NaN,0.326223,0.000163,NaN,NaN,0.579340,0.000252,NaN,NaN
3,2021-11-05 08:00:30,train,57.335069,1228.762976,58.463541,3031.794189,59.5,0.154282,0.0,80.946182,81.81424,50.422452,19.470506,6.357205,0.326340,0.578588,1229.088501,1229.049451,1229.492151,NaN,58.234953,58.272569,58.408563,NaN,3038.494727,3052.526001,3054.785938,NaN,59.5,59.5,59.5,NaN,0.154225,0.154282,0.154167,NaN,0.0,0.0,0.0,NaN,80.946182,80.954863,80.976565,NaN,81.81424,81.81424,81.82292,NaN,50.434027,50.422452,50.439814,NaN,19.315793,19.102801,18.861438,NaN,6.325521,6.293403,6.266928,NaN,0.326403,0.326181,0.326085,NaN,0.579051,0.579456,0.579514,NaN,1228.966976,0.177745,NaN,NaN,58.323687,0.122568,NaN,NaN,3040.938306,10.579712,NaN,NaN,0.154263,0.000033,NaN,NaN,80.949076,5.011819e-03,NaN,NaN,81.814240,5.553572e-09,NaN,NaN,0.326308,0.000115,NaN,NaN,0.579032,0.000434,NaN,NaN
4,2021-11-05 08:00:40,train,57.835647,1229.

In [23]:
# Revisamos tamaño antes de eliminar nulos generados por lags y rolling

filas_antes_dropna = df_features.shape[0]

print("Filas antes de dropna:", filas_antes_dropna)
print("Columnas:", df_features.shape[1])
print("Nulos totales:", df_features.isnull().sum().sum())

Filas antes de dropna: 23070
Columnas: 100
Nulos totales: 804


In [24]:
# Eliminamos filas con nulos generados por lags y rolling

df_features_limpio = df_features.dropna().copy()

In [25]:
# Revisamos filas por bloque después de dropna

df_features_limpio.groupby("bloque").agg(
    inicio=("Time", "min"),
    fin=("Time", "max"),
    filas=("Time", "count")
)

,inicio,fin,filas
bloque,,,
test,2021-11-07 08:01:00,2021-11-08 00:04:50,5784
train,2021-11-05 08:01:00,2021-11-06 07:59:50,8634
validacion,2021-11-06 08:01:00,2021-11-07 07:59:50,8634


In [27]:
# Guardamos el dataset preparado para el Modelo A

df_features_limpio.to_csv("data_limpio/features_modelo_A.csv", index=False)